In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

cat = "`data-bricks-fundamentals`.default"

In [0]:
from pyspark.sql import functions as F

cat = "`data-bricks-fundamentals`.default"

geos = ["Canada", "Newfoundland and Labrador", "Prince Edward Island", "Nova Scotia",
        "New Brunswick", "Quebec", "Ontario", "Manitoba", "Saskatchewan",
        "Alberta", "British Columbia"]

# ---------- CPI: 32 food groups, Canada + 10 provinces, 2017 onward ----------
food_groups = [
    "Food", "Food purchased from stores", "Food purchased from restaurants",
    "Meat", "Fresh or frozen meat (excluding poultry)", "Fresh or frozen beef",
    "Fresh or frozen pork", "Fresh or frozen poultry", "Fresh or frozen chicken",
    "Processed meat", "Fish, seafood and other marine products", "Fish",
    "Dairy products and eggs", "Dairy products", "Fresh milk", "Butter", "Cheese", "Eggs",
    "Bakery and cereal products (excluding baby food)", "Bakery products",
    "Cereal products (excluding baby food)",
    "Fruit, fruit preparations and nuts", "Fresh fruit",
    "Preserved fruit and fruit preparations",
    "Vegetables and vegetable preparations", "Fresh vegetables",
    "Preserved vegetables and vegetable preparations",
    "Other food products and non-alcoholic beverages", "Edible fats and oils",
    "Sugar and confectionery", "Coffee and tea", "Non-alcoholic beverages",
]

silver_cpi = (spark.table(f"{cat}.bronze_cpi")
    .filter(F.col("geo").isin(geos))
    .filter(F.col("products_and_product_groups").isin(food_groups))
    .withColumn("month", F.to_date("ref_date", "yyyy-MM"))
    .filter(F.col("month") >= F.lit("2017-01-01"))
    .select("month", "geo",
            F.col("products_and_product_groups").alias("product"),
            F.col("value").alias("cpi_index"))
)
silver_cpi.write.mode("overwrite").saveAsTable(f"{cat}.silver_cpi_food")

# ---------- Average prices: food products only, Canada + 10 provinces ----------
non_food = ["Deodorant, 85 grams", "Toothpaste, 100 millilitres",
            "Shampoo, 400 millilitres", "Laundry detergent, 4.43 litres"]

silver_avg = (spark.table(f"{cat}.bronze_avg_prices")
    .filter(F.col("geo").isin(geos))
    .withColumn("product", F.trim("products"))          # some names have trailing spaces
    .filter(~F.col("product").isin(non_food))
    .withColumn("month", F.to_date("ref_date", "yyyy-MM"))
    .filter(F.col("value").isNotNull())
    .select("month", "geo", "product", F.col("value").alias("avg_price"))
)
silver_avg.write.mode("overwrite").saveAsTable(f"{cat}.silver_avg_prices_food")

In [0]:
print(spark.table(f"{cat}.silver_cpi_food").count())         # expect 41,184
print(spark.table(f"{cat}.silver_avg_prices_food").count())  # expect 135,140